In [3]:
import duckdb
import pandas as pd

In [6]:
conn = duckdb.connect('../lebron_analytics.duckdb', read_only= True)

In [7]:
print("=" * 70)
print("LEBRON SHOT DATA EXPLORATION")
print("=" * 70)

LEBRON SHOT DATA EXPLORATION


In [ ]:
# 1. Table info
print("\nTABLE: raw.lebron_shots")
result = conn.execute("""
        SELECT COUNT(*) as total_rows,
               COUNT(DISTINCT season) as total_seasons,
               MIN(game_date) as earliest_game,
               MAX(game_date) as latest_game
        FROM raw.lebron_shots
    """).fetchone()


TABLE: raw.lebron_shots


In [9]:
print(f"  Total shots: {result[0]:,}")
print(f"  Seasons: {result[1]}")
print(f"  Date range: {result[2]} to {result[3]}")

  Total shots: 31,100
  Seasons: 23
  Date range: 20031029 to 20260203


In [10]:
columns = conn.execute("""
select column_name, data_type
                       from information_schema.columns
                       where table_schema = 'raw'
                       and table_name = 'lebron_shots'
                       order by ordinal_position
""").fetchall()

In [11]:
for i, (col_name, col_type) in enumerate(columns, 1):
        print(f"  {i:2}. {col_name:<30} {col_type}")


   1. GRID_TYPE                      VARCHAR
   2. GAME_ID                        VARCHAR
   3. GAME_EVENT_ID                  BIGINT
   4. PLAYER_ID                      BIGINT
   5. PLAYER_NAME                    VARCHAR
   6. TEAM_ID                        BIGINT
   7. TEAM_NAME                      VARCHAR
   8. PERIOD                         BIGINT
   9. MINUTES_REMAINING              BIGINT
  10. SECONDS_REMAINING              BIGINT
  11. EVENT_TYPE                     VARCHAR
  12. ACTION_TYPE                    VARCHAR
  13. SHOT_TYPE                      VARCHAR
  14. SHOT_ZONE_BASIC                VARCHAR
  15. SHOT_ZONE_AREA                 VARCHAR
  16. SHOT_ZONE_RANGE                VARCHAR
  17. SHOT_DISTANCE                  DOUBLE
  18. LOC_X                          DOUBLE
  19. LOC_Y                          DOUBLE
  20. SHOT_ATTEMPTED_FLAG            BIGINT
  21. SHOT_MADE_FLAG                 BIGINT
  22. GAME_DATE                      VARCHAR
  23. HTM            

In [ ]:
  # 3. Sample data
print("\nSAMPLE DATA (5 random shots):")
sample = conn.execute("""
        SELECT * 
        FROM raw.lebron_shots 
        USING SAMPLE 5
    """).fetchdf()
    
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 30)
print(sample.to_string(index=False))


SAMPLE DATA (5 random shots):
        GRID_TYPE    GAME_ID  GAME_EVENT_ID  PLAYER_ID  PLAYER_NAME    TEAM_ID           TEAM_NAME  PERIOD  MINUTES_REMAINING  SECONDS_REMAINING  EVENT_TYPE        ACTION_TYPE      SHOT_TYPE SHOT_ZONE_BASIC       SHOT_ZONE_AREA SHOT_ZONE_RANGE  SHOT_DISTANCE  LOC_X  LOC_Y  SHOT_ATTEMPTED_FLAG  SHOT_MADE_FLAG GAME_DATE HTM VTM  season                                                                 source_file
Shot Chart Detail 0020300573            312       2544 LeBron James 1610612739 Cleveland Cavaliers       3                  7                 16 Missed Shot         Layup Shot 2PT Field Goal Restricted Area            Center(C) Less Than 8 ft.            0.0    0.0    0.0                    1               0  20040117 UTA CLE 2003-04 C:\Users\Ashim\Desktop\lebron_shot-analytics\data\raw\shots_2003-04.parquet
Shot Chart Detail 0021100024            452       2544 LeBron James 1610612748          Miami Heat       4                  4                 34 

In [ ]:
   # 4. Shots per season
print("\nSHOTS PER SEASON:")
season_stats = conn.execute("""
        SELECT 
            season,
            COUNT(*) as shots,
            SUM(CASE WHEN shot_made_flag = 1 THEN 1 ELSE 0 END) as made,
            ROUND(AVG(CASE WHEN shot_made_flag = 1 THEN 1.0 ELSE 0.0 END) * 100, 1) as fg_pct
        FROM raw.lebron_shots
        GROUP BY season
        ORDER BY season
    """).fetchall()


SHOTS PER SEASON:


In [14]:
for season, shots, made, fg_pct in season_stats:
        print(f"  {season}: {shots:>4} shots | {made:>4} made | {fg_pct:>5}% FG")

  2003-04: 1492 shots |  622 made |  41.7% FG
  2004-05: 1684 shots |  795 made |  47.2% FG
  2005-06: 1823 shots |  875 made |  48.0% FG
  2006-07: 1621 shots |  772 made |  47.6% FG
  2007-08: 1642 shots |  794 made |  48.4% FG
  2008-09: 1613 shots |  789 made |  48.9% FG
  2009-10: 1528 shots |  768 made |  50.3% FG
  2010-11: 1485 shots |  758 made |  51.0% FG
  2011-12: 1169 shots |  621 made |  53.1% FG
  2012-13: 1354 shots |  765 made |  56.5% FG
  2013-14: 1353 shots |  767 made |  56.7% FG
  2014-15: 1279 shots |  624 made |  48.8% FG
  2015-16: 1416 shots |  737 made |  52.0% FG
  2016-17: 1344 shots |  736 made |  54.8% FG
  2017-18: 1580 shots |  857 made |  54.2% FG
  2018-19: 1095 shots |  558 made |  51.0% FG
  2019-20: 1303 shots |  643 made |  49.3% FG
  2020-21:  823 shots |  422 made |  51.3% FG
  2021-22: 1221 shots |  640 made |  52.4% FG
  2022-23: 1219 shots |  609 made |  50.0% FG
  2023-24: 1269 shots |  685 made |  54.0% FG
  2024-25: 1270 shots |  651 made 

In [ ]:
 # 5. Check for important columns
print("\nKEY COLUMN CHECKS:")
    
    # Check if common columns exist
common_cols_actual = [
    'SHOT_MADE_FLAG', 'SHOT_TYPE', 'SHOT_ZONE_BASIC', 
    'SHOT_ZONE_AREA', 'SHOT_DISTANCE', 'PERIOD', 
    'MINUTES_REMAINING', 'SECONDS_REMAINING', 'ACTION_TYPE',
    'SHOT_ZONE_RANGE'
]
    
existing_cols = [col[0] for col in columns]
    
for col in common_cols_actual:
        exists = '[ok]' if col in existing_cols else '[--]'
        print(f"  {exists} {col}")


KEY COLUMN CHECKS:
  [ok] SHOT_MADE_FLAG
  [ok] SHOT_TYPE
  [ok] SHOT_ZONE_BASIC
  [ok] SHOT_ZONE_AREA
  [ok] SHOT_DISTANCE
  [ok] PERIOD
  [ok] MINUTES_REMAINING
  [ok] SECONDS_REMAINING
  [ok] ACTION_TYPE
  [ok] SHOT_ZONE_RANGE


In [ ]:
    # 6. Missing values
print("\nMISSING VALUES CHECK:")
for col_name, _ in columns[:15]:  # Check first 15 columns
    null_count = conn.execute(f"""
            SELECT COUNT(*) 
            FROM raw.lebron_shots 
            WHERE {col_name} IS NULL
        """).fetchone()[0]
        
    if null_count > 0:
        pct = (null_count / result[0]) * 100
        print(f"  {col_name}: {null_count:,} nulls ({pct:.1f}%)")


MISSING VALUES CHECK:
  SHOT_ZONE_BASIC: 2 nulls (0.0%)
  SHOT_ZONE_AREA: 2 nulls (0.0%)
